# 02 Data cleaning pipeline

> **Synthetic data, sampled simulation.** 768 simulated runs compared with 19,708 scheduled runs (3.9%). Not IRCTC or Indian Railways network-wide data. No forecasts. Associations are not causes.

**Purpose.** Show the reproducible Pandas pipeline in `analytics/cleaning.py`: type conversion, date handling, duplicate detection, missing-value handling, category normalisation, derived columns, validation flags and aggregation-ready datasets.

**Rules of the pipeline:** no row is removed, nothing is imputed, every rule is logged with the number of rows it touched, and source files are never modified (everything works on copies).

In [1]:
# Setup: find the project root, make python/src importable, load shared helpers (the data is only READ)
import sys, pathlib, warnings
warnings.filterwarnings("ignore")
ROOT = next(p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents] if (p / "data" / "processed" / "cleaned").exists())
sys.path.insert(0, str(ROOT / "python" / "src"))
import pandas as pd, numpy as np
pd.set_option("display.width", 220); pd.set_option("display.max_columns", 40); pd.set_option("display.float_format", lambda v: f"{v:,.4f}")
from analytics import data_loader as dl, validation, cleaning, kpi, eda, SYNTHETIC_NOTE
from analytics import statistics as stats
print(SYNTHETIC_NOTE)

Synthetic data, sampled simulation (768 simulated runs of 19,708 scheduled). Not IRCTC or Indian Railways network-wide data; no forecasts.


In [2]:
ds = dl.load_dataset()
clean = cleaning.run_cleaning(ds.tables)

## 1. The cleaning log

One row per rule application (`reports/phase5_cleaning_log.csv`).

In [3]:
display(clean.log)

,rule_id,table,description,rows_in,rows_out,rows_affected,action
0,C-01,stations,trim leading/trailing whitespace in text columns,32.0000,32,0,values trimmed; no rows removed
1,C-01,routes,trim leading/trailing whitespace in text columns,40.0000,40,0,values trimmed; no rows removed
2,C-01,trains,trim leading/trailing whitespace in text columns,52.0000,52,0,values trimmed; no rows removed
3,C-01,train_stops,trim leading/trailing whitespace in text columns,203.0000,203,0,values trimmed; no rows removed
4,C-01,calendar,trim leading/trailing whitespace in text columns,975.0000,975,0,values trimmed; no rows removed
5,C-01,seat_capacity,trim leading/trailing whitespace in text columns,"2,285.0000",2285,0,values trimmed; no rows removed
6,C-01,train_operations,trim leading/trailing whitespace in text columns,768.0000,768,0,values trimmed; no rows removed
7,C-01,bookings,trim leading/trailing whitespace in text columns,"201,182.0000",201182,0,values trimmed; no rows removed
8,C-01,booking_passengers,trim leading/trailing whitespace in text columns,"405,200.0000",405200,0,values trimmed; no rows removed
9,C-01,cancellations,trim leading/trailing whitespace in text columns,"43,923.0000",43923,0,values trimmed; no rows removed


## 2. Rows in equals rows out

In [4]:
cmp = pd.DataFrame({'rows_in': {k: len(v) for k, v in ds.tables.items()}, 'rows_out': {k: len(v) for k, v in clean.tables.items()}})
cmp['difference'] = cmp.rows_out - cmp.rows_in
display(cmp)
assert (cmp.difference == 0).all()

,rows_in,rows_out,difference
stations,32,32,0
routes,40,40,0
trains,52,52,0
train_stops,203,203,0
calendar,975,975,0
seat_capacity,2285,2285,0
train_operations,768,768,0
bookings,201182,201182,0
booking_passengers,405200,405200,0
cancellations,43923,43923,0


## 3. Type conversion and category normalisation

Statuses, classes and age bands become ordered categoricals so tables and charts use a consistent order.

In [5]:
ps = clean.tables['booking_passengers']
print(ps.final_status.dtype, list(ps.final_status.cat.categories))
print(clean.tables['bookings'].travel_class.dtype, list(clean.tables['bookings'].travel_class.cat.categories))

category ['Confirmed', 'RAC', 'Waitlisted', 'Cancelled']
category ['1A', '2A', '3A', 'SL', 'CC', '2S']


## 4. Duplicate detection (demonstrated on a COPY)

The cleaned data has no duplicates (Phase 2 removed the injected ones). To show the rule works, three rows are duplicated in an in-memory copy; the source is untouched.

In [6]:
bk = ds.tables['bookings']
test = pd.concat([bk, bk.head(3)], ignore_index=True)
print('duplicates in the source data :', int(bk.duplicated('booking_id').sum()))
print('duplicates detected in the copy:', int(test.duplicated('booking_id').sum()))

duplicates in the source data : 0
duplicates detected in the copy: 3


## 5. Missing values: retained and flagged

`arrival_delay_min` NULL (23 runs) and `age_band` = 'Unknown' (1,010 passengers) are kept as they are; flag columns record them.

In [7]:
rf0 = clean.facts['run_fact']; print('runs with a missing arrival delay:', int(rf0.delay_missing.sum()), '| on_time is NULL for them:', int(rf0.on_time.isna().sum()))
print('passengers with age Unknown:', int(clean.facts['passenger_fact'].age_missing.sum()))

runs with a missing arrival delay: 23 | on_time is NULL for them: 23
passengers with age Unknown: 1010


## 6. Derived columns

In [8]:
display(clean.facts['booking_fact'][['booking_id', 'journey_date', 'journey_month', 'weekday', 'is_weekend', 'season', 'is_peak', 'lead_days', 'lead_bucket', 'booking_status', 'n_pax', 'n_cancelled']].head(8))
display(clean.facts['passenger_fact'][['passenger_id', 'initial_status', 'final_status', 'is_cancelled', 'is_initial_wl', 'is_cleared_wl', 'is_uncleared_wl', 'seat_equivalent']].head(8))

,booking_id,journey_date,journey_month,weekday,is_weekend,season,is_peak,lead_days,lead_bucket,booking_status,n_pax,n_cancelled
0,1000001,2024-09-01,2024-09,Sunday,True,Monsoon,False,119,61-120 days,Confirmed,1,0
1,1000002,2024-09-02,2024-09,Monday,False,Monsoon,False,120,61-120 days,Confirmed,2,0
2,1000003,2024-09-03,2024-09,Tuesday,False,Monsoon,False,119,61-120 days,Confirmed,2,0
3,1000004,2024-09-04,2024-09,Wednesday,False,Monsoon,False,119,61-120 days,Cancelled,1,1
4,1000005,2024-09-02,2024-09,Monday,False,Monsoon,False,115,61-120 days,Confirmed,3,0
5,1000006,2024-09-03,2024-09,Tuesday,False,Monsoon,False,116,61-120 days,Confirmed,1,0
6,1000007,2024-09-02,2024-09,Monday,False,Monsoon,False,114,61-120 days,Confirmed,2,0
7,1000008,2024-09-02,2024-09,Monday,False,Monsoon,False,114,61-120 days,Confirmed,1,0


,passenger_id,initial_status,final_status,is_cancelled,is_initial_wl,is_cleared_wl,is_uncleared_wl,seat_equivalent
0,1,Confirmed,Confirmed,False,False,False,False,1.0000
1,2,Confirmed,Confirmed,False,False,False,False,1.0000
2,3,Confirmed,Confirmed,False,False,False,False,1.0000
3,4,Confirmed,Confirmed,False,False,False,False,1.0000
4,5,Confirmed,Confirmed,False,False,False,False,1.0000
5,6,Confirmed,Cancelled,True,False,False,False,0.0000
6,7,Confirmed,Confirmed,False,False,False,False,1.0000
7,8,Confirmed,Confirmed,False,False,False,False,1.0000


## 7. Aggregation-ready datasets

In [9]:
display(pd.DataFrame({k: {'rows': len(v), 'columns': v.shape[1]} for k, v in clean.facts.items()}).T)
display(clean.facts['class_run_fact'][['capacity_id', 'journey_month', 'travel_class', 'total_seats', 'passengers', 'confirmed_pax', 'rac_pax', 'wl_final_pax', 'booked_seats', 'occupancy', 'demand_per_seat']].head(6))

,rows,columns
passenger_fact,405200,31
booking_fact,201182,31
event_fact,43923,16
class_run_fact,2285,30
run_fact,768,32


,capacity_id,journey_month,travel_class,total_seats,passengers,confirmed_pax,rac_pax,wl_final_pax,booked_seats,occupancy,demand_per_seat
0,1,2024-09,3A,192,238,171,33,0,187.5000,0.9766,1.2396
1,2,2024-09,SL,216,139,116,0,0,116.0000,0.5370,0.6435
2,3,2024-09,2S,324,496,322,0,96,322.0000,0.9938,1.5309
3,4,2024-09,2A,92,104,82,11,0,87.5000,0.9511,1.1304
4,5,2024-09,3A,192,145,135,0,0,135.0000,0.7031,0.7552
5,6,2024-09,SL,216,276,192,46,0,215.0000,0.9954,1.2778
